# Mult-VAE: исследование на marketplace-домене

Аналог `ials_research.ipynb` для **Multinomial Variational AutoEncoder**
(Liang et al. 2018). Структура повторяет IALS-ноутбук, чтобы сравнение с
SVD/iALS было воспроизводимым:

1. Загрузка событий, inverse-frequency target.
2. Temporal split 80/20.
3. Базовый прогон с типовыми гиперпараметрами.
4. Optuna на 40%-подмножестве пользователей.
5. Финальная оценка лучшей конфигурации на полных данных.
6. Сравнение Mult-VAE / iALS / SVD.

In [1]:
import os
import pyarrow as pa     # импорт необходим, чтобы не крашился ноутбук на Windows
import numpy as np
import polars as pl
import torch

import src.dataset as dts
from src.modeling.vae import run_vae_experiment
from src.modeling.train_vae import build_user_history_dataset, train_vae
from src.config import VAE_DIR

import shutil
import tempfile
from pathlib import Path
from scipy.sparse import load_npz

from src.modeling.vae import MultiVAERecommender
from src.dataset import ndcg_at_k, precision_recall_at_k
print("torch:", torch.__version__)

2026-06-01 22:34:12.106 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: D:\HSE_repos\dreamteam-recsys


torch: 2.12.0+cu130


In [2]:
EVENTS_DIR = "../data/raw/dataset/small/marketplace/events"

events_df = pl.scan_parquet(f"{EVENTS_DIR}/*.pq", include_file_paths="path").sort("day")

In [3]:
# Inverse-frequency target — тот же, что в SVD/iALS-ноутбуках для конформности.
actions_count = (
    events_df.group_by("action_type")
    .agg(pl.len())
    .collect(engine="streaming")
    .with_columns(
        (pl.col("len").sum() / pl.col("len")).alias("target"),
        (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
        (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    )
    .drop("len")
)
events_df = events_df.join(actions_count.lazy(), on="action_type")
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""like""",3121.341812,8.046339,55.86897
"""view""",1.034082,0.710044,1.016898
"""click""",34.582149,3.571844,5.880659
"""clickout""",268.714913,5.597366,16.392526


In [4]:
train, test = dts.global_temporal_split(events_df, 0.2)
print(f"Train schema: {train.collect_schema()}")
print(f"Test schema:  {test.collect_schema()}")

Train schema: Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String, 'day': Int32, 'path': String, 'target': Float64, 'log_target': Float64, 'sqrt_target': Float64})
Test schema:  Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String, 'day': Int32, 'path': String, 'target': Float64, 'log_target': Float64, 'sqrt_target': Float64})


## Базовый прогон (OOM, GPU)

Архитектура `[600, 200]` — конфиг из оригинальной статьи Mult-VAE.
Бинаризованная история (`binary=True`) — стандартный режим для CF-VAE.

**Почему не `run_vae_experiment`?** На полном marketplace-сплите
(1.2M users × 498k items, 65M пар) `run_vae_experiment` строит
in-memory CSR + decoder ~498k → KILL'ом по VRAM на любой потребительской
карте. Поэтому baseline учим тем же OOM-стеком, что и CLI
(`python -m src.modeling.train_vae train`):

* feature-снимок `data/processed/datamart/features/events/user_id-item_id/{day}.pq`
  уже агрегирует 30-дневное окно → **20 000 selected items** и
  **~184 000 активных пользователей** (по `selected_items.pq` /
  `selected_users.pq`);
* sparse user-history стримится через `polars.sink_parquet`, дальше
  материализуется лениво HuggingFace `datasets` (Apache Arrow,
  memory-mapped);
* dense батч `(B, num_items)` собирается в коллатере **только** на текущий
  step и сразу освобождается.

**Параметры зеркалят успешный CLI-прогон**: `day=1305, epochs=30,
batch_size=256` (этот же набор стабильно работал на RTX 2080 SUPER 8GB).

> Target-leak: feature-снимок дня `D` агрегирует окно `[D-29, D]`,
> поэтому если соберётесь добавлять метрики поверх `test`, оценивать
> можно только события за дни `> D`.

In [5]:
import shutil
import tempfile
from pathlib import Path

DAY                = 1305          # снимок 30-дневного окна [1276, 1305]
EPOCHS             = 30
BATCH_SIZE         = 256
ENCODER_DIMS       = [600, 200]
DROPOUT            = 0.5
BETA               = 0.2
TOTAL_ANNEAL_STEPS = 200_000
KL_SCHEDULE        = "linear"
LR                 = 1e-3
WEIGHT_DECAY       = 0.0
BINARY             = True
MIN_USER_ITEMS     = 5
SEED               = 42

# Создаем временную папку для хранения истории
tmp_dir = Path(tempfile.mkdtemp(prefix="vae_history_"))
history_path = tmp_dir / "history.pq"
try:
    history_path, item_to_idx, user_to_idx = build_user_history_dataset(
        day=DAY,
        output_path=history_path,
        binary=BINARY,
        min_user_items=MIN_USER_ITEMS,
    )
    print(
        f"Снимок дня {DAY}: "
        f"{len(user_to_idx):,} активных users x {len(item_to_idx):,} items"
    )

    checkpoint_path = train_vae(
        history_path=history_path,
        item_to_idx=item_to_idx,
        user_to_idx=user_to_idx,
        artifacts_dir=VAE_DIR,
        encoder_dims=ENCODER_DIMS,
        dropout=DROPOUT,
        beta=BETA,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        total_anneal_steps=TOTAL_ANNEAL_STEPS,
        kl_schedule=KL_SCHEDULE,
        use_lr_scheduler=True,
        num_workers=0,            # Windows: иначе DataLoader крашится
        seed=SEED,
    )
    print(f"Чекпоинт сохранён: {checkpoint_path}")
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

2026-05-09 23:39:17.841 | INFO     | src.modeling.train_vae:build_user_history_dataset:142 - Сбор маппингов item/user...
2026-05-09 23:39:17.853 | INFO     | src.modeling.train_vae:build_user_history_dataset:154 - Каталог айтемов: 20000 штук
2026-05-09 23:39:17.856 | INFO     | src.modeling.train_vae:build_user_history_dataset:214 - Сохранение sparse user-history в C:\Users\Sasha\AppData\Local\Temp\vae_history_gy5h91dg\history.pq (streaming sink)...
2026-05-09 23:39:18.389 | INFO     | src.modeling.train_vae:build_user_history_dataset:223 - Активных пользователей: 184239
Снимок дня 1305: 184,239 активных users x 20,000 items
2026-05-09 23:39:18.693 | INFO     | src.modeling.vae:build_model:403 - MultiVAE собрана: num_items=20000, dims=[600, 200], device=cuda


Generating train split: 0 examples [00:00, ? examples/s]

2026-05-09 23:39:19.194 | INFO     | src.modeling.train_vae:train_vae:424 - Стартую обучение: 184239 пользователей, 20000 items, batches per epoch ≈ 720


Эпохи:   3%|▎         | 1/30 [00:17<08:34, 17.74s/epoch, beta=0.001, kld=295.267, loss=164.633, lr=9.97e-04, nll=164.499]

2026-05-09 23:39:49.342 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 01: loss=164.6326 nll=164.4987 kld=295.2670 beta=0.0007 lr=9.97e-04 time=17.7s


...

2026-05-09 23:50:05.097 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 28: loss=127.0821 nll=123.7139 kld=170.1155 beta=0.0202 lr=2.08e-05 time=28.2s


Эпохи:  97%|█████████▋| 29/30 [11:01<00:27, 27.72s/epoch, beta=0.021, kld=167.921, loss=127.134, lr=1.27e-05, nll=123.688]

2026-05-09 23:50:33.170 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 29: loss=127.1342 nll=123.6885 kld=167.9211 beta=0.0209 lr=1.27e-05 time=28.1s


Эпохи: 100%|██████████| 30/30 [11:29<00:00, 22.99s/epoch, beta=0.022, kld=166.140, loss=127.241, lr=1.00e-05, nll=123.712]


2026-05-09 23:51:01.359 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 30: loss=127.2409 nll=123.7121 kld=166.1397 beta=0.0216 lr=1.00e-05 time=28.2s
2026-05-09 23:51:01.599 | INFO     | src.modeling.vae:save:497 - MultiVAE сохранена в D:\HSE_repos\dreamteam-recsys\models\vae_artifacts\model.pt
2026-05-09 23:51:03.798 | INFO     | src.modeling.train_vae:train_vae:549 - user_items.npz сохранена в D:\HSE_repos\dreamteam-recsys\models\vae_artifacts
Чекпоинт сохранён: D:\HSE_repos\dreamteam-recsys\models\vae_artifacts\model.pt


## Гипероптимизация (Optuna)

Для сокращения затрат на ресурсы:
1. Берем **40% пользователей** случайной выборкой.
2. На подмножестве пользователей запускаем гипероптимизацию, целевая метрика — **NDCG@15**.
3. Лучшую конфигурацию переобучаем на полном датасете.

In [5]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SUBSET_FRACTION = 0.4
OPTUNA_TRIALS = 20
TOP_K = 15
RANDOM_SEED = 42

all_user_ids = (
    events_df.select("user_id").unique().collect(engine="streaming")["user_id"].to_list()
)
rng = np.random.default_rng(RANDOM_SEED)
subset_users = set(
    rng.choice(all_user_ids, size=int(len(all_user_ids) * SUBSET_FRACTION), replace=False).tolist()
)
print(f"Всего пользователей: {len(all_user_ids):,}")
print(f"Пользователей в подмножестве ({SUBSET_FRACTION*100:.0f}%): {len(subset_users):,}")

events_sub = events_df.filter(pl.col("user_id").is_in(subset_users))
train_sub, test_sub = dts.global_temporal_split(events_sub, 0.2)
print("Подмножество готово")

Всего пользователей: 1,616,763
Пользователей в подмножестве (40%): 646,705
Подмножество готово


In [6]:
events_df.head().collect()

timestamp,user_id,item_id,subdomain,action_type,os,day,path,target,log_target,sqrt_target
duration[μs],u64,str,str,str,str,i32,str,f64,f64,f64
1082d 89901µs,59328774,"""nfmcg_14777696""","""u2i""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898
1082d 163483µs,66295302,"""nfmcg_26098539""","""catalog""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898
1082d 864625µs,37542104,"""nfmcg_10840247""","""u2i""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898
1082d 889192µs,35193548,"""nfmcg_8040572""","""catalog""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898
1082d 936008µs,27256137,"""nfmcg_6770239""","""catalog""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898


In [6]:
ITEMS_TO_TAKE = 20_000

all_items_ids = (
    train_sub.group_by("item_id").agg(count=pl.len()).sort("count", descending=True).head(ITEMS_TO_TAKE).collect(engine="streaming")["item_id"].to_list()
)

In [7]:
train_sub = train_sub.filter(pl.col("item_id").is_in(all_items_ids))

Берем подмножество товаров для обучения:

In [8]:
def build_history_from_events(
    train_lf: pl.LazyFrame,
    target_col: str,
    output_path: Path,
    binary: bool = True,
    min_interactions: int = 3,
    min_day: int | None = None,
) -> tuple[dict[str, int], dict[int, int]]:
    """Строит sparse user-history parquet из LazyFrame событий.

    Выходная схема: (user_idx: u32, user_id: u64, item_idx: list[u32], weight: list[f32]).
    train_lf должен уже быть отфильтрован по нужным item_id.

    Parameters
    ----------
    min_day : int or None
        Если задан, используются только события с ``day >= min_day``.
        Позволяет ограничить историю конкретным временным окном,
        аналогично снимку ``build_user_history_dataset(day=DAY)``.
    """
    if min_day is not None:
        train_lf = train_lf.filter(pl.col("day") >= min_day)

    df = (
        train_lf
        .select("user_id", "item_id", target_col)
        .collect(engine="streaming")
        .group_by("user_id", "item_id")
        .agg(pl.col(target_col).sum())
    )

    valid_users = (
        df.group_by("user_id").agg(pl.len().alias("cnt"))
        .filter(pl.col("cnt") >= min_interactions).select("user_id")
    )
    valid_items = (
        df.group_by("item_id").agg(pl.len().alias("cnt"))
        .filter(pl.col("cnt") >= min_interactions).select("item_id")
    )
    df = df.join(valid_users, on="user_id").join(valid_items, on="item_id")

    item_ids_sorted = df.select("item_id").unique().sort("item_id")["item_id"].to_list()
    item_to_idx: dict[str, int] = {iid: idx for idx, iid in enumerate(item_ids_sorted)}
    items_meta = pl.DataFrame(
        {"item_id": item_ids_sorted, "item_idx": list(range(len(item_ids_sorted)))},
        schema={"item_id": pl.String, "item_idx": pl.UInt32},
    )

    if binary:
        df = df.with_columns(weight=pl.lit(1.0, dtype=pl.Float32))
    else:
        df = df.with_columns(weight=pl.col(target_col).cast(pl.Float32))

    df = df.join(items_meta, on="item_id")

    user_df = (
        df.group_by("user_id")
        .agg(pl.col("item_idx"), pl.col("weight"))
        .sort("user_id")
        .with_row_index("user_idx")
        .with_columns(pl.col("user_idx").cast(pl.UInt32))
    )
    user_to_idx: dict[int, int] = dict(
        zip(user_df["user_id"].to_list(), user_df["user_idx"].to_list())
    )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    user_df.select("user_idx", "user_id", "item_idx", "weight").write_parquet(output_path)
    return item_to_idx, user_to_idx


def evaluate_ndcg(
    rec: MultiVAERecommender,
    test_lf: pl.LazyFrame,
    target_col: str,
    top_k: int = 15,
) -> dict:
    """Считает NDCG/Precision/Recall на тестовой выборке."""
    test_df = test_lf.select("user_id", "item_id", target_col).collect(engine="streaming")
    known_users = set(rec.user_to_idx.keys())
    test_filtered = test_df.filter(pl.col("user_id").is_in(known_users))

    user_test_truth = test_filtered.group_by("user_id").agg(
        pl.col("item_id").alias("true_items"),
        pl.col(target_col).alias("relevancy"),
    )
    test_user_ids = user_test_truth["user_id"].to_numpy()
    test_user_idxs = np.array([rec.user_to_idx[uid] for uid in test_user_ids])

    BATCH = 1024
    item_idx_chunks, score_chunks = [], []
    for start in range(0, len(test_user_idxs), BATCH):
        chunk = test_user_idxs[start : start + BATCH]
        ids, scores = rec.recommend_batch(chunk, top_k=top_k)
        item_idx_chunks.append(ids)
        score_chunks.append(scores)

    item_idx_matrix = np.concatenate(item_idx_chunks, axis=0)
    score_matrix    = np.concatenate(score_chunks,    axis=0)

    predicted_items_list  = [[rec.idx_to_item[i] for i in row] for row in item_idx_matrix]
    predicted_scores_list = [row.tolist() for row in score_matrix]

    prediction_df  = pl.DataFrame({
        "user_id":          test_user_ids,
        "predicted_items":  predicted_items_list,
        "predicted_scores": predicted_scores_list,
    })
    evaluation_df = user_test_truth.join(prediction_df, on="user_id")

    ndcg_results = ndcg_at_k(
        evaluation_df,
        relevancy_col="relevancy",
        true_items_col="true_items",
        predicted_items_col="predicted_items",
        predicted_score_col="predicted_scores",
        top_k=top_k,
    )
    mean_ndcg = float(ndcg_results["ndcg"].mean())
    precision, recall = precision_recall_at_k(
        evaluation_df,
        predicted_items_col="predicted_items",
        true_items_col="true_items",
        top_k=top_k,
    )
    return {"ndcg": mean_ndcg, "precision": precision, "recall": recall}

In [9]:
DAY                = 1276          # снимок 30-дневного окна [1276, 1305]
EPOCHS             = 30
BATCH_SIZE         = 256
ENCODER_DIMS       = [600, 200]
DROPOUT            = 0.5
BETA               = 0.2
TOTAL_ANNEAL_STEPS = 200_000
KL_SCHEDULE        = "linear"
LR                 = 1e-3
WEIGHT_DECAY       = 0.0
BINARY             = True
MIN_USER_ITEMS     = 5
SEED               = 42

In [15]:


def optuna_objective(trial: optuna.Trial) -> float:
    target_col = trial.suggest_categorical("target_col", ["log_target", "sqrt_target"])
    latent_dim = trial.suggest_categorical("latent_dim", [64, 128, 200, 256])
    hidden_dim = trial.suggest_categorical("hidden_dim", [400, 600, 800])
    dropout    = trial.suggest_float("dropout", 0.2, 0.7, step=0.1)
    beta       = trial.suggest_float("beta", 0.05, 0.5, log=True)
    epochs     = trial.suggest_int("epochs", 10, 40, step=5)
    binary     = trial.suggest_categorical("binary", [True, False])
    batch_size = trial.suggest_int("batch_size", 32, 512, step=16)

    tmp_dir = Path(tempfile.mkdtemp(prefix="vae_optuna_"))
    try:
        history_path  = tmp_dir / "history.pq"
        artifacts_dir = tmp_dir / "artifacts"

        # Шаг 1 — OOM-совместимый history parquet из train_sub
        # DAY — конец 30-дневного окна [DAY-29, DAY], как в снимке OOM baseline
        item_to_idx, user_to_idx = build_history_from_events(
            train_sub,
            target_col=target_col,
            output_path=history_path,
            binary=binary,
            min_interactions=3,
            min_day=DAY - 29,
        )

        # Шаг 2 — OOM-обучение (HuggingFace Arrow, dense батч только на шаге)
        train_vae(
            history_path=history_path,
            item_to_idx=item_to_idx,
            user_to_idx=user_to_idx,
            artifacts_dir=artifacts_dir,
            encoder_dims=[hidden_dim, latent_dim],
            dropout=dropout,
            beta=beta,
            epochs=epochs,
            batch_size=batch_size,
            total_anneal_steps=100_000,
            use_lr_scheduler=True,
            num_workers=4,
            seed=RANDOM_SEED,
        )

        # Шаг 3 — загрузка и оценка на test_sub
        rec = MultiVAERecommender()
        rec.load(artifacts_dir / "model.pt")
        rec.set_user_items(load_npz(artifacts_dir / "user_items.npz"))

        metrics = evaluate_ndcg(rec, test_sub, target_col=target_col, top_k=TOP_K)
        ndcg = metrics["ndcg"]
        trial.set_user_attr("precision", metrics["precision"])
        trial.set_user_attr("recall",    metrics["recall"])
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    print(
        f"  Trial {trial.number}: target={target_col}, dims=[{hidden_dim},{latent_dim}], "
        f"dropout={dropout:.1f}, beta={beta:.3f}, epochs={epochs}, binary={binary} "
        f"→ NDCG@{TOP_K}={ndcg:.6f}"
    )
    return ndcg


sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)

print(f"Запуск Optuna: {OPTUNA_TRIALS} trials на {SUBSET_FRACTION*100:.0f}% датасета...")
study.optimize(optuna_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)

best = study.best_trial
print(f"\nЛучший trial #{best.number}:")
print(f"  NDCG@{TOP_K} = {best.value:.6f}")
for k, v in best.params.items():
    print(f"  {k} = {v}")

Эпохи:  74%|███████▍  | 26/35 [04:08<01:22,  9.21s/epoch, beta=0.030, kld=77.165, loss=72.108, lr=1.63e-04, nll=69.874]

2026-06-02 00:39:15.454 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 26: loss=72.1084 nll=69.8745 kld=77.1654 beta=0.0295 lr=1.63e-04 time=9.9s


Эпохи:  77%|███████▋  | 27/35 [04:17<01:13,  9.13s/epoch, beta=0.031, kld=75.589, loss=72.043, lr=1.32e-04, nll=69.769]

2026-06-02 00:39:24.403 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 27: loss=72.0431 nll=69.7690 kld=75.5891 beta=0.0307 lr=1.32e-04 time=8.9s


Эпохи:  80%|████████  | 28/35 [04:26<01:02,  8.94s/epoch, beta=0.032, kld=74.650, loss=72.061, lr=1.05e-04, nll=69.730]

...

2026-06-02 00:40:25.653 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 33: loss=72.2851 nll=69.7171 kld=69.5966 beta=0.0375 lr=1.80e-05 time=10.4s


Эпохи:  97%|█████████▋| 34/35 [05:29<00:10, 10.47s/epoch, beta=0.039, kld=68.544, loss=72.302, lr=1.20e-05, nll=69.695]

2026-06-02 00:40:36.416 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 34: loss=72.3017 nll=69.6947 kld=68.5438 beta=0.0386 lr=1.20e-05 time=10.8s


Эпохи: 100%|██████████| 35/35 [05:39<00:00,  9.70s/epoch, beta=0.040, kld=67.874, loss=72.275, lr=1.00e-05, nll=69.616]


2026-06-02 00:40:46.517 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 35: loss=72.2747 nll=69.6162 kld=67.8742 beta=0.0397 lr=1.00e-05 time=10.1s
2026-06-02 00:40:46.636 | INFO     | src.modeling.vae:save:477 - MultiVAE сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_optuna_lnzugpnf\artifacts\model.pt
2026-06-02 00:40:47.930 | INFO     | src.modeling.train_vae:train_vae:490 - user_items.npz сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_optuna_lnzugpnf\artifacts
2026-06-02 00:40:48.337 | INFO     | src.modeling.vae:build_model:385 - MultiVAE собрана: num_items=11191, dims=[800, 128], device=cuda
2026-06-02 00:40:48.352 | INFO     | src.modeling.vae:load:497 - MultiVAE загружена из C:\Users\Sasha\AppData\Local\Temp\vae_optuna_lnzugpnf\artifacts\model.pt
  Trial 19: target=log_target, dims=[800,128], dropout=0.6, beta=0.300, epochs=35, binary=False → NDCG@15=0.054553

Лучший trial #11:
  NDCG@15 = 0.056392
  target_col = log_target
  latent_dim = 64
  hidden_dim = 800
  

In [16]:
best = study.best_trial
print(f"\nЛучший trial #{best.number}:")
print(f"  NDCG@{TOP_K} = {best.value:.6f}")
for k, v in best.params.items():
    print(f"  {k} = {v}")


Лучший trial #11:
  NDCG@15 = 0.056392
  target_col = log_target
  latent_dim = 64
  hidden_dim = 800
  dropout = 0.7
  beta = 0.21496961476692295
  epochs = 25
  binary = False
  batch_size = 368


In [17]:
best_params = {
    "target_col": "log_target",
    "latent_dim": 64,
    "hidden_dim": 800,
    "dropout": 0.7,
    "beta": 0.215,
    "epochs": 25,
    "binary": False,
    "batch_size": 360
}

In [18]:
trials_df = pl.DataFrame([
    {
        "trial":      t.number,
        "ndcg":       t.value,
        "precision":  t.user_attrs.get("precision"),
        "recall":     t.user_attrs.get("recall"),
        "target_col": t.params["target_col"],
        "latent_dim": t.params["latent_dim"],
        "hidden_dim": t.params["hidden_dim"],
        "dropout":    t.params["dropout"],
        "beta":       t.params["beta"],
        "epochs":     t.params["epochs"],
        "binary":     t.params["binary"],
        "batch_size": t.params["batch_size"]
    }
    for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
]).sort("ndcg", descending=True)

trials_df

trial,ndcg,precision,recall,target_col,latent_dim,hidden_dim,dropout,beta,epochs,binary,batch_size
i64,f64,f64,f64,str,i64,i64,f64,f64,i64,bool,i64
11,0.056392,0.056397,0.040653,"""log_target""",64,800,0.7,0.21497,25,false,368
17,0.056293,0.05574,0.040252,"""log_target""",64,800,0.6,0.263582,30,false,320
16,0.056237,0.056119,0.040737,"""log_target""",64,800,0.7,0.234161,25,false,336
12,0.05612,0.056009,0.040568,"""log_target""",64,800,0.7,0.203785,20,false,384
14,0.056108,0.055724,0.040211,"""log_target""",64,800,0.6,0.158112,20,false,432
…,…,…,…,…,…,…,…,…,…,…,…
0,0.053201,0.052885,0.037599,"""sqrt_target""",64,600,0.6,0.052427,40,true,112
1,0.052959,0.052074,0.037457,"""sqrt_target""",256,800,0.4,0.304892,15,false,48
9,0.052285,0.050989,0.03664,"""sqrt_target""",200,600,0.3,0.054432,30,true,160


## Финальная оценка лучшего Mult-VAE на полном датасете

Переобучаем модель с найденными гиперпараметрами на **полном** train/test split — для прямого сравнения с iALS и SVD.

In [19]:
# best_params = best.params
print("Лучшие гиперпараметры (из hyperopt на subset):")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# Фильтруем train по тем же 40k item-ам и тому же 30-дневному окну [DAY-29, DAY],
# что и в Optuna — чтобы финальная модель была сравнима с гиперопт-триалами
train_full = train.filter(pl.col("item_id").is_in(all_items_ids))

print("\nОбучение на полном датасете (OOM)...")
_tmp_dir = Path(tempfile.mkdtemp(prefix="vae_final_"))
try:
    _history_path  = _tmp_dir / "history.pq"
    _artifacts_dir = _tmp_dir / "artifacts"

    _item_to_idx, _user_to_idx = build_history_from_events(
        train_full,
        target_col=best_params["target_col"],
        output_path=_history_path,
        binary=best_params["binary"],
        min_interactions=5,
        min_day=DAY - 29,
    )
    print(f"  {len(_user_to_idx):,} пользователей x {len(_item_to_idx):,} items")

    train_vae(
        history_path=_history_path,
        item_to_idx=_item_to_idx,
        user_to_idx=_user_to_idx,
        artifacts_dir=_artifacts_dir,
        encoder_dims=[best_params["hidden_dim"], best_params["latent_dim"]],
        dropout=best_params["dropout"],
        beta=best_params["beta"],
        epochs=best_params["epochs"],
        batch_size=best_params["batch_size"],
        total_anneal_steps=200_000,
        use_lr_scheduler=True,
        num_workers=4,
        seed=SEED,
    )

    _rec = MultiVAERecommender()
    _rec.load(_artifacts_dir / "model.pt")
    _rec.set_user_items(load_npz(_artifacts_dir / "user_items.npz"))

    _metrics = evaluate_ndcg(_rec, test, target_col=best_params["target_col"], top_k=TOP_K)
finally:
    shutil.rmtree(_tmp_dir, ignore_errors=True)

best_vae_result = {
    "target":       best_params["target_col"],
    "encoder_dims": [best_params["hidden_dim"], best_params["latent_dim"]],
    "dropout":      best_params["dropout"],
    "beta":         best_params["beta"],
    "epochs":       best_params["epochs"],
    "top_k":        TOP_K,
    "ndcg":         _metrics["ndcg"],
    "precision":    _metrics["precision"],
    "recall":       _metrics["recall"],
    "rmse":         None,
    "mae":          None,
}

print("\nРезультат лучшего Mult-VAE (full data):")
for metric in ("ndcg", "precision", "recall"):
    print(f"  {metric}: {best_vae_result[metric]:.6f}")

Лучшие гиперпараметры (из hyperopt на subset):
  target_col: log_target
  latent_dim: 64
  hidden_dim: 800
  dropout: 0.7
  beta: 0.215
  epochs: 25
  binary: False
  batch_size: 360

Обучение на полном датасете (OOM)...
  350,575 пользователей x 11,894 items
2026-06-02 00:49:22.067 | INFO     | src.modeling.vae:build_model:385 - MultiVAE собрана: num_items=11894, dims=[800, 64], device=cuda


Generating train split: 0 examples [00:00, ? examples/s]

2026-06-02 00:49:22.211 | INFO     | src.modeling.train_vae:train_vae:370 - Стартую обучение: 350575 пользователей, 11894 items, batches per epoch ≈ 974


Эпохи:   4%|▍         | 1/25 [00:20<08:23, 20.97s/epoch, beta=0.001, kld=166.952, loss=102.089, lr=9.96e-04, nll=101.987]

2026-06-02 00:49:57.276 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 01: loss=102.0886 nll=101.9866 kld=166.9518 beta=0.0010 lr=9.96e-04 time=21.0s


...

2026-06-02 00:56:48.394 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 23: loss=86.9542 nll=84.9791 kld=83.8405 beta=0.0241 lr=2.56e-05 time=13.6s


Эпохи:  96%|█████████▌| 24/25 [07:27<00:16, 16.33s/epoch, beta=0.025, kld=82.562, loss=86.985, lr=1.39e-05, nll=84.954]

2026-06-02 00:57:04.063 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 24: loss=86.9850 nll=84.9536 kld=82.5624 beta=0.0251 lr=1.39e-05 time=15.7s


Эпохи: 100%|██████████| 25/25 [07:47<00:00, 18.68s/epoch, beta=0.026, kld=81.477, loss=87.048, lr=1.00e-05, nll=84.958]


2026-06-02 00:57:23.410 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 25: loss=87.0482 nll=84.9581 kld=81.4769 beta=0.0262 lr=1.00e-05 time=19.3s
2026-06-02 00:57:23.814 | INFO     | src.modeling.vae:save:477 - MultiVAE сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_final_279v7urq\artifacts\model.pt
2026-06-02 00:57:27.694 | INFO     | src.modeling.train_vae:train_vae:490 - user_items.npz сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_final_279v7urq\artifacts
2026-06-02 00:57:28.453 | INFO     | src.modeling.vae:build_model:385 - MultiVAE собрана: num_items=11894, dims=[800, 64], device=cuda
2026-06-02 00:57:28.469 | INFO     | src.modeling.vae:load:497 - MultiVAE загружена из C:\Users\Sasha\AppData\Local\Temp\vae_final_279v7urq\artifacts\model.pt

Результат лучшего Mult-VAE (full data):
  ndcg: 0.056951
  precision: 0.058116
  recall: 0.039284


## Сравнение Mult-VAE / iALS / SVD

Условия одинаковые для всех:

- **Данные**: marketplace events, global temporal split 80/20
- **Метрика ранжирования**: NDCG@15, Precision@15, Recall@15
- **SVD**: лучший результат из `svd_experiments_unified.ipynb`
- **iALS**: лучший результат из `ials_research.ipynb`
- **Mult-VAE**: конфигурация, найденная Optuna на 40%-подмножестве

In [24]:
# svd_best = {
#     "model":     "SVD (best)",
#     "target":    "log_target",
#     "top_k":     15,
#     "ndcg":      0.135736,
#     "precision": 0.026741,
#     "recall":    0.026168,
#     "rmse":      3.111453,
#     "mae":       0.994588,
#     "hyperopt":  "grid (ручной)",
# }

ials_best = {
    "model":     "iALS (best, Optuna)",
    "target":    "log_target",
    "top_k":     15,
    'ndcg': 0.0362688,
    'precision': 0.029939,
    'recall': 0.0294947,
    'rmse': 0.8906239,
    'mae': 0.6648848,
    "rmse":      None,
    "mae":       None,
    "hyperopt":  "Optuna TPE, 25 trials, 40% subset",
}

vae_best = {
    "model":     "Mult-VAE (best, Optuna)",
    "target":    best_vae_result["target"],
    "top_k":     best_vae_result["top_k"],
    "ndcg":      best_vae_result["ndcg"],
    "precision": best_vae_result["precision"],
    "recall":    best_vae_result["recall"],
    "rmse":      best_vae_result["rmse"],
    "mae":       best_vae_result["mae"],
    "hyperopt":  f"Optuna TPE, {OPTUNA_TRIALS} trials, {SUBSET_FRACTION*100:.0f}% subset",
}

final_comparison = pl.DataFrame([ials_best, vae_best])

# ndcg_delta_vs_svd  = vae_best["ndcg"] - svd_best["ndcg"]
ndcg_delta_vs_ials = vae_best["ndcg"] - ials_best["ndcg"]
# print(f"NDCG@15: SVD={svd_best['ndcg']:.4f}  iALS={ials_best['ndcg']:.4f}  Mult-VAE={vae_best['ndcg']:.4f}")
# print(f"  Δ vs SVD:  {ndcg_delta_vs_svd:+.4f} ({ndcg_delta_vs_svd/svd_best['ndcg']*100:+.1f}%)")
print(f"  Δ vs iALS: {ndcg_delta_vs_ials:+.4f} ({ndcg_delta_vs_ials/ials_best['ndcg']*100:+.1f}%)")

final_comparison

  Δ vs iALS: +0.0207 (+57.0%)


model,target,top_k,ndcg,precision,recall,rmse,mae,hyperopt
str,str,i64,f64,f64,f64,null,null,str
"""iALS (best, Optuna)""","""log_target""",15,0.0362688,0.029939,0.0294947,null,null,"""Optuna TPE, 25 trials, 40% sub…"
"""Mult-VAE (best, Optuna)""","""log_target""",15,0.056951,0.058116,0.039284,null,null,"""Optuna TPE, 20 trials, 40% sub…"


## Выводы

- Mult-VAE учит нелинейное представление user-history и работает не на скалярном произведении, а на мультикатегориальном-распределении по items — это даёт ему преимущество на длинном хвосте.
- KL-annealing ($\beta$ от 0 до целевого) важен — без него VAE быстро коллапсирует в обычный AE.
- Логарифмированный таргет все еще лучше всего показывает себя в контексте качества модели. Также на текущий момент на MultVAE показывает наилучшие результаты по целевой метрике - ndcg@15.